---

### 🎓 **Professor**: Apostolos Filippas

### 📘 **Class**: Web Analytics

### 📋 **Topic**: Learning a Non-linear Model

🚫 **Note**: You are not allowed to share the contents of this notebook with anyone outside this class without written permission by the professor.

---

> **Before you begin:** [Update your course files and Python environment safely.](https://github.com/apostolosfilippas/wa/blob/main/assets/before-you-begin.md)



# 1. 🌳 Learning a Decision Tree

In the previous notebook, we used linear regression to predict a car's MPG. Let's use the SAME car data to learn a different kind of model: a decision tree!

This time, instead of predicting the exact MPG, we'll predict whether a car belongs to a high-MPG or low-MPG group. More on this soon.

The packages below are already included in the course environment. If needed, run `uv sync` in your terminal in the `wa` folder, then select the course's `.venv` Python environment as your notebook kernel.

Run the cells in order; later cells use things we created earlier.

## 1.1 Getting and cleaning our data

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

%matplotlib inline
sns.set_theme(style='ticks', palette='Set2')

# notebooks can start in wa, scripts, or homework; let's find the course folder
course_folder = Path.cwd().resolve()
if course_folder.name in {'scripts', 'homework'}:
    course_folder = course_folder.parent
if not (course_folder / 'pyproject.toml').is_file():
    raise RuntimeError('Open the wa course folder in VS Code, then restart this notebook.')

data_folder = course_folder / 'data'
print('Course folder:', course_folder)

We'll use the SAME [Auto MPG dataset from UCI](https://archive.ics.uci.edu/dataset/9/auto+mpg) as in Lecture 7. A copy is already in our `data` folder, so there's nothing to download.

We'll also use the same five predictors. As before, we leave out cars with a missing predictor or MPG. This is a simple starting point, not a rule that we should delete every row with ANY missing value!

In [ ]:
column_names = ['mpg', 'cylinders', 'displacement', 'horsepower', 'weight', 'acceleration',
                'model_year', 'origin', 'car_name']

# this file separates values with spaces, rather than commas
mpg_df = pd.read_csv(data_folder / 'auto-mpg.data-original',
                     sep=r'\s+',
                     header=None,
                     names=column_names)

# the same five predictors we used in the linear-regression notebook
predictor_cols = ['weight', 'acceleration', 'horsepower', 'cylinders', 'displacement']

print('Cars before cleaning:', len(mpg_df))
mpg_df = mpg_df.dropna(subset=predictor_cols + ['mpg']).copy()
print('Cars we can use:', len(mpg_df))

mpg_df.head()

## 1.2 Turning the regression problem into a classification problem

In regression, we predict a number, like a car's exact MPG. In classification, we predict which group (or **class**) it belongs to.

Let's turn our MPG problem into a classification problem: will a car have high or low MPG?

First, let's look at how MPG varies across our cars.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
mpg_df['mpg'].hist(bins=15, color='#4C72B0', edgecolor='white', ax=ax)
ax.set_title('MPG in our car data')
ax.set_xlabel('Miles per gallon (MPG)')
ax.set_ylabel('Number of cars')
fig.tight_layout()
plt.show()

Arbitrarily, let's say that cars with MPG **greater than the median** belong to the high-MPG class.

The median is the middle value when we put MPG in order. We'll use `1` for high MPG and `0` for low MPG. A car at or below the median gets `0`.

These labels are relative to OUR dataset, not a universal definition of good fuel economy!

In [ ]:
# find and print the median
median_mpg = mpg_df['mpg'].median()
print(f'The median MPG is: {median_mpg:.2f}')

# this function turns one MPG value into a class: 1 or 0
def is_high_mpg(mpg):
    return 1 if mpg > median_mpg else 0

# apply() runs our function on every value in the mpg column
# we save the result as a NEW column, keeping the original mpg
mpg_df['is_high_mpg'] = mpg_df['mpg'].apply(is_high_mpg)

mpg_df[['car_name', 'mpg', 'is_high_mpg']].head(10)

Our algorithm's task is now to predict which class a car belongs to.

We will give it information like weight and horsepower. The answer we want it to learn is `is_high_mpg`, NOT the original MPG.

Why not give it MPG as one of the predictors? Because we used MPG to create the answer! That would give the model the very information we are trying to predict.

## 1.3 What's a decision tree?

The best way to learn is to read Chapter 3 from "Data Science for Business" by Provost and Fawcett. Nevertheless, we'll try to give you a brief overview here.

A **decision tree**, as the name suggests, can be visualized as a tree. It asks a series of questions about the information we give it, then uses the answers to make a prediction.

Decision trees can be used for both classification and regression. For our purposes, we'll only look at classification.

### Components of a decision tree

- **Root node**: The starting point, before any questions have been answered. All our training cars start here.
- **Decision nodes**: Ask a question about one feature, like "Is the car's weight less than this number?" The answer tells us which branch to follow.
- **Leaf nodes**: The endpoints. There are no more questions; the leaf gives us a predicted class.

### How does a decision tree learn?

We don't write all these questions ourselves! When we call `.fit()`, the algorithm tries different features and cutoffs to find useful ways to separate the classes.

For example, a useful question might divide a mixed group of cars into one group that is mostly high MPG and another that is mostly low MPG. The algorithm repeats this process within the new groups.

Unlike a linear model, a tree doesn't use one fixed coefficient per predictor everywhere. It can use different rules in different parts of the data. That's why we're calling it a **non-linear model**.

### A simple example

Let's consider a simple decision-making scenario: deciding if we should play tennis based on the weather.

- The root node may ask: "Is it raining?"
    - If Yes, we might conclude, "Do not play tennis." This becomes a leaf.
    - If No, we might ask: "Is the humidity high?"
        - If Yes, "Do not play tennis."
        - If No, "Play tennis."

This example just illustrates the idea. With our cars, the tree will learn its questions from the data.

## 1.4 Building a decision tree

We'll use scikit-learn, just like we did for linear regression.

Again, **X** contains our predictors and **y** contains the answer we want to predict. Here, y is a class (`0` or `1`), rather than the exact MPG.

Look at these arguments when we create the tree:

- **`max_depth=3`**: A car can go through at most THREE questions on its way from the root to a leaf. This keeps our tree small enough to read.
- **`criterion='entropy'`**: Tells the tree how to compare possible splits. Entropy measures how mixed the classes are. A group with only one class has entropy 0; a mixed group has higher entropy.
- **`random_state=42`**: Makes the result repeatable with the same data and settings. The number 42 isn't special; we just keep it fixed.

In [ ]:
from sklearn.tree import DecisionTreeClassifier

# the SAME five predictors as in our linear-regression notebook
X = mpg_df[predictor_cols]
y = mpg_df['is_high_mpg']

# create a tree; it hasn't learned anything yet!
decision_tree = DecisionTreeClassifier(
    max_depth=3,
    criterion='entropy',
    random_state=42
)

# fit() learns questions and cutoffs using X and the known classes in y
decision_tree.fit(X, y)

## 1.5 Looking at our tree

We now have a fitted tree. Let's see what it learned!

Scikit-learn can draw it directly with matplotlib. We don't need to install Graphviz, use a terminal command, or create image files.

In [ ]:
from sklearn.tree import plot_tree

fig, ax = plt.subplots(figsize=(14, 8))
nodes = plot_tree(
    decision_tree,
    feature_names=predictor_cols,
    class_names=['Low MPG', 'High MPG'],
    proportion=True,
    rounded=True,
    precision=2,
    fontsize=11,
    ax=ax
)

# make the node boxes easy to see with our plotting style
for node in nodes:
    if node.get_bbox_patch() is not None:
        node.get_bbox_patch().set(edgecolor='#666666', facecolor='white')

ax.set_title('Decision tree: Low MPG vs. High MPG', pad=18)
fig.tight_layout()
plt.show()

Check out the decision tree. Start at the top!

- The first question it "asks" is whether displacement is **<= 190.5**.
    - If True, go LEFT. The next question is whether horsepower is **<= 84.5**.
    - If False, go RIGHT. The next question is whether acceleration is **<= 15.9**.

Keep following the answers until you reach a leaf. Its `class` is our prediction for that car.

### What do the other numbers mean?

- **`entropy`**: How mixed the classes are at this node. Zero means all the cars there belong to the same class. At the root, entropy is 1 because our two classes are equally common.
- **`samples`**: The percentage of ALL our training cars that reach this node. Every car starts at the root, so the root shows 100%.
- **`value`**: The class proportions WITHIN this node, in the order **[Low MPG, High MPG]**. For example, `[0.2, 0.8]` would mean 20% low MPG and 80% high MPG.
- **`class`**: The most common class at that node. It appears in the other boxes too, but we keep following questions until we reach a leaf.

In other words, the tree is a series of nested "if" statements. Much like the "Guess who?" game or [Akinator](https://en.akinator.com/theme-selection)!

---
# ❓ 2. Are our predictions good?


How good is our model? Let's start by getting its predictions for the cars we used to fit it.

Notice: these are the SAME cars whose answers we already showed the model. We're checking how well it learned the training data, not how well it will do on new cars.

In [ ]:
# predict() uses our fitted tree to assign a class to each car
preds = decision_tree.predict(X)

# keep the actual class and the prediction next to each other
predictions_mpg_df = mpg_df.assign(predictions=preds)
predictions_mpg_df[['car_name', 'mpg', 'is_high_mpg', 'predictions']].head(10)

We'll use **accuracy**: the fraction of cars whose predicted class matches the actual class.

Both kinds of correct answer count: correctly predicting low MPG AND correctly predicting high MPG.

It's easy to compute this yourselves, or you can use scikit-learn's `accuracy_score`. We give it the actual answers first, then the predictions.

In [ ]:
from sklearn.metrics import accuracy_score

training_accuracy = accuracy_score(y, preds)
print(f'Training accuracy: {training_accuracy:.1%}')

Is this good? Should we be satisfied with this level of accuracy?

Not so fast! This is **training accuracy**. Looking good on cars we've already shown the model does NOT tell us how well it will predict cars it hasn't seen.

We ended the linear-regression notebook with the same warning. It applies to decision trees too!

In the [next notebook](08.2%20evaluating-models.ipynb), we'll look at overfitting and how to evaluate a model on data it did NOT use for learning :)